In [9]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import pandas as pd
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE = "https://sahaexpo.com"
LIST_URL = "https://sahaexpo.com/en/exhibitors"

headers = {"User-Agent": "Mozilla/5.0"}


# =========================
# CLEAN EMOJI
# =========================
def remove_emoji(text):
    return re.sub(r"[^\x00-\x7F]+", "", text).strip()


# =========================
# GET LINKS (FAST)
# =========================
def get_links():
    links = set()
    page = 1

    while True:
        url = f"{LIST_URL}?page={page}"
        print(f"Scraping page {page}...")

        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        cards = soup.select("a.exhibitor-card")

        if not cards:
            break

        for a in cards:
            href = a.get("href")
            links.add(urljoin(BASE, href))

        page += 1
        time.sleep(1)

    return list(links)


# =========================
# INIT SELENIUM (ONCE)
# =========================
def init_driver():
    options = Options()
    options.headless = True
    driver = webdriver.Firefox(options=options)
    return driver


# =========================
# SCRAPE DETAIL (SELENIUM)
# =========================
def scrape_detail(driver, url):
    driver.get(url)

    wait = WebDriverWait(driver, 10)

    data = {
        "name": None,
        "industry": None,
        "email": None,
        "website": None,
        "address": None,
        "country": None,
        "hall": None,
        "stand": None,
        "about": None,
        "url": url
    }

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # NAME
    h1 = soup.select_one(".container-saha > h1")
    if h1:
        data["name"] = h1.text.strip()

    # INDUSTRY
    industries = soup.select(".exhibitor-categories-sidebar .exhibitor-category-tag-sm")
    if industries:
        data["industry"] = ", ".join([i.text.strip() for i in industries])

    # EMAIL 
    try:
        email_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href^='mailto:']"))
        )
        data["email"] = email_el.get_attribute("href").replace("mailto:", "")
    except:
        pass

    # WEBSITE
    for a in soup.select("a.exhibitor-contact-item[href^='http']"):
        href = a.get("href")

        if any(x in href for x in ["linkedin", "twitter", "wa.me"]):
            continue

        if "sahaexpo.com" not in href:
            data["website"] = href
            break

    # ADDRESS & COUNTRY
    for div in soup.select(".exhibitor-contact-list .exhibitor-contact-item"):
        if div.name == "div":
            text = div.text.strip()

            if len(text.split()) <= 3:
                data["country"] = remove_emoji(text)
            else:
                data["address"] = text

    # HALL / STAND
    loc = soup.select_one(".exhibitor-location-inline")
    if loc:
        text = loc.text

        hall = re.search(r"Hall:\s*(\d+)", text)
        stand = re.search(r"Stand:\s*([A-Z0-9\-a-z]+)", text)

        if hall:
            data["hall"] = hall.group(1)
        if stand:
            data["stand"] = stand.group(1)

    # ABOUT
    about = soup.select_one(".exhibitor-about-content")
    if about:
        data["about"] = about.text.strip()

    return data


# =========================
# MAIN
# =========================
def main():
    links = get_links()
    print("Total:", len(links))

    driver = init_driver()
    results = []

    for i, url in enumerate(links):
        print(f"[{i+1}] {url}")

        try:
            data = scrape_detail(driver, url)

            print("   Email:", data["email"])
            results.append(data)

        except Exception as e:
            print("Error:", e)

        time.sleep(1)

    driver.quit()

    df = pd.DataFrame(results)
    df.to_csv("saha_exhibitors_2026.csv", index=False)

    print("Done!")


if __name__ == "__main__":
    main()

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...
Scraping 

[84] https://sahaexpo.com/en/katilimci/moment-teknoloji
   Email: moment@momentas.com.tr
[85] https://sahaexpo.com/en/katilimci/lanxess-diphyl
   Email: sales@mabayco.com
[86] https://sahaexpo.com/en/katilimci/datatel-telemetry
   Email: sales@datatel-telemetry.de
[87] https://sahaexpo.com/en/katilimci/telestream
   Email: USSales@telestream.net
[88] https://sahaexpo.com/en/katilimci/enderus
   Email: sales@enderus.com
[89] https://sahaexpo.com/en/katilimci/derinsu-savunma
   Email: info@derinsudefence.com.tr
[90] https://sahaexpo.com/en/katilimci/atapars
   Email: info@ataparsdefence.com
[91] https://sahaexpo.com/en/katilimci/smac
   Email: contact@smac.fr
[92] https://sahaexpo.com/en/katilimci/qualwawe
   Email: ida@ida-test.com
[93] https://sahaexpo.com/en/katilimci/deltav
   Email: info@deltav.com.tr
[94] https://sahaexpo.com/en/katilimci/avs
   Email: info@avstekstil.com.tr
[95] https://sahaexpo.com/en/katilimci/samsor
   Email: ss@samsor.cn
[96] https://sahaexpo.com/en/katilimci/

   Email: software@materialise.be
[184] https://sahaexpo.com/en/katilimci/tdk-teknoloji
   Email: info@tdkteknoloji.com.tr
[185] https://sahaexpo.com/en/katilimci/era-rf
   Email: info@erarf.com
[186] https://sahaexpo.com/en/katilimci/bsc-filters
   Email: kim.large@bscfilters.com
[187] https://sahaexpo.com/en/katilimci/ekaytech
   Email: info@ekaytech.com.tr
[188] https://sahaexpo.com/en/katilimci/quanser
   Email: sales@enox.com.tr
[189] https://sahaexpo.com/en/katilimci/antenova
   Email: info@e-kom.com
[190] https://sahaexpo.com/en/katilimci/velocity-one
   Email: info@velocity-one.com
[191] https://sahaexpo.com/en/katilimci/t-motor-turkiye
   Email: onlinesales@tmotor.com
[192] https://sahaexpo.com/en/katilimci/kuartis
   Email: info@kuartis.com
[193] https://sahaexpo.com/en/katilimci/weintek
   Email: info@ozdisan.com
[194] https://sahaexpo.com/en/katilimci/mfm-muhendislik
   Email: mfm@mfmmuhendislik.com
[195] https://sahaexpo.com/en/katilimci/trelleborg-sealing
   Email: info.t

[283] https://sahaexpo.com/en/katilimci/cukurova
   Email: export@cumitas.com
[284] https://sahaexpo.com/en/katilimci/nyco-sa
   Email: info@nyco-group.com
[285] https://sahaexpo.com/en/katilimci/makner
   Email: info@maknerteknoloji.com
[286] https://sahaexpo.com/en/katilimci/mdc-savunma
   Email: info@mdcdefence.com
[287] https://sahaexpo.com/en/katilimci/rams-global
   Email: None
[288] https://sahaexpo.com/en/katilimci/quatro-metal
   Email: info@quatrometal.com
[289] https://sahaexpo.com/en/katilimci/shieldtechnic
   Email: info@adozmuhendislik.com
[290] https://sahaexpo.com/en/katilimci/tc-sanayi-ve-teknoloji-bakanligi
   Email: info@sanayi.gov.tr
[291] https://sahaexpo.com/en/katilimci/apem-sas
   Email: allen.homonoff@apem.com
[292] https://sahaexpo.com/en/katilimci/novanta-zettlex
   Email: info@novanta.com
[293] https://sahaexpo.com/en/katilimci/rescue-trailer-sro
   Email: petr.skocovsky@rescuetrailer.cz
[294] https://sahaexpo.com/en/katilimci/hidromode
   Email: oktaybdd@hi

[382] https://sahaexpo.com/en/katilimci/hagat
   Email: info@hagat.com.tr
[383] https://sahaexpo.com/en/katilimci/ems
   Email: ems@emselektronik.com.tr
[384] https://sahaexpo.com/en/katilimci/aselsan-elektronik-sanayi-ve-ticaret-as
   Email: ahacioglu@aselsan.com
[385] https://sahaexpo.com/en/katilimci/milshel
   Email: info@milshel.net
[386] https://sahaexpo.com/en/katilimci/locosys
   Email: same@sameas.com.tr
[387] https://sahaexpo.com/en/katilimci/titco
   Email: ck@titco.com.tr
[388] https://sahaexpo.com/en/katilimci/arbor-technology
   Email: sales@arbor.com.tw
[389] https://sahaexpo.com/en/katilimci/deltav-dis-alan
   Email: info@deltav.com.tr
[390] https://sahaexpo.com/en/katilimci/demmerg
   Email: info@demmerg.com
[391] https://sahaexpo.com/en/katilimci/meggit
   Email: aerospace@ch.meggitt.com
[392] https://sahaexpo.com/en/katilimci/mostas
   Email: info@mostastech.com
[393] https://sahaexpo.com/en/katilimci/innophase-iot
   Email: info@e-kom.com
[394] https://sahaexpo.com/

...

[1074] https://sahaexpo.com/en/katilimci/ocean-voyager
   Email: info@oceanvoyagermarine.com
[1075] https://sahaexpo.com/en/katilimci/seaward
   Email: sales@seaward.com
[1076] https://sahaexpo.com/en/katilimci/lightning-power
   Email: info@lightning.com.tr
[1077] https://sahaexpo.com/en/katilimci/laversab
   Email: sales@laversab.com
[1078] https://sahaexpo.com/en/katilimci/emzer
   Email: ida@ida-test.com
[1079] https://sahaexpo.com/en/katilimci/fsmvu-aluteam
   Email: aluteam@fsm.edu.tr
[1080] https://sahaexpo.com/en/katilimci/trtest
   Email: info@tr-test.com.tr
[1081] https://sahaexpo.com/en/katilimci/schloder
   Email: ida@ida-test.com
[1082] https://sahaexpo.com/en/katilimci/harting
   Email: info@harting.com
[1083] https://sahaexpo.com/en/katilimci/karma-kodlama-mak-san-tic-ltd-sti
   Email: info@karmakodlama.com.tr
[1084] https://sahaexpo.com/en/katilimci/atakel
   Email: info@atakel.com
[1085] https://sahaexpo.com/en/katilimci/altinkaya
   Email: satis@altinkaya.com
[1086] h

   Email: info@atlasdry.com
[1171] https://sahaexpo.com/en/katilimci/br-vision
   Email: info@adozmuhendislik.com
[1172] https://sahaexpo.com/en/katilimci/xingto
   Email: victor@xtbattery.com
[1173] https://sahaexpo.com/en/katilimci/argemsan
   Email: argemsan@argemsan.com
[1174] https://sahaexpo.com/en/katilimci/ceramicspeed
   Email: industry@ceramicspeed.com
[1175] https://sahaexpo.com/en/katilimci/go-light
   Email: info@adozmuhendislik.com
[1176] https://sahaexpo.com/en/katilimci/nicomatic
   Email: turkey@nicomatic.com
[1177] https://sahaexpo.com/en/katilimci/chargebyte
   Email: info@adozmuhendislik.com
[1178] https://sahaexpo.com/en/katilimci/medef-defence
   Email: info@medefence.net
[1179] https://sahaexpo.com/en/katilimci/corvus
   Email: info@corvus.tech
[1180] https://sahaexpo.com/en/katilimci/iveo
   Email: info@iveoelectronics.com
[1181] https://sahaexpo.com/en/katilimci/vaunix
   Email: ida@ida-test.com
[1182] https://sahaexpo.com/en/katilimci/enetki
   Email: bilgi@en

[1271] https://sahaexpo.com/en/katilimci/ayvos
   Email: halil.sahin@ayvos.com
[1272] https://sahaexpo.com/en/katilimci/egebimtes
   Email: egebimtes@egebimtes.com.tr
[1273] https://sahaexpo.com/en/katilimci/precizika-metrology-uab
   Email: r.zmitroviciute@precizika.com
[1274] https://sahaexpo.com/en/katilimci/ssi
   Email: info@ssi.gov.tr
[1275] https://sahaexpo.com/en/katilimci/spacelis
   Email: info@spacelis.com
[1276] https://sahaexpo.com/en/katilimci/intersonik-makina
   Email: info@intersonik.com
[1277] https://sahaexpo.com/en/katilimci/skylink-fpv
   Email: 2885385159@qq.com
[1278] https://sahaexpo.com/en/katilimci/alfa-decor
   Email: info@alfadecor.com
[1279] https://sahaexpo.com/en/katilimci/cinetech
   Email: info@ozdisan.com
[1280] https://sahaexpo.com/en/katilimci/visage
   Email: sales@visageint.org
[1281] https://sahaexpo.com/en/katilimci/gokser-arge
   Email: trade@gokser.com
[1282] https://sahaexpo.com/en/katilimci/motor-g
   Email: None
[1283] https://sahaexpo.com/e

   Email: scanco@scanivalve.com
[1370] https://sahaexpo.com/en/katilimci/atlas-expo
   Email: info@atlas-expo.com.tr
[1371] https://sahaexpo.com/en/katilimci/miltas
   Email: kesicitakim@miltas.com.tr
[1372] https://sahaexpo.com/en/katilimci/supaero
   Email: saha@supaero.de
[1373] https://sahaexpo.com/en/katilimci/mdh-makel
   Email: info@mdhmakel.com
[1374] https://sahaexpo.com/en/katilimci/eae-elektroteknik
   Email: info@eaegroup.com
[1375] https://sahaexpo.com/en/katilimci/tse
   Email: halklailiskiler@tse.org.tr
[1376] https://sahaexpo.com/en/katilimci/teknohab
   Email: mustafa.acarl@teknohab.com.tr
[1377] https://sahaexpo.com/en/katilimci/tubitak-ume
   Email: ume@tubitak.gov.tr
[1378] https://sahaexpo.com/en/katilimci/heico
   Email: sales@heico.com
[1379] https://sahaexpo.com/en/katilimci/taloshub
   Email: emre@taloshub.com
[1380] https://sahaexpo.com/en/katilimci/bmc-dis-alan
   Email: info@bmc.com.tr
[1381] https://sahaexpo.com/en/katilimci/hursan
   Email: hursan@hursanpr

[1467] https://sahaexpo.com/en/katilimci/damise
   Email: info@damise.com
[1468] https://sahaexpo.com/en/katilimci/spacrol
   Email: info@spacrol.com
Done!


In [10]:
import pandas as pd

df = pd.read_csv('saha_exhibitors_2026.csv')
df

,name,industry,email,website,address,country,hall,stand,about,url
0,KZ MEKATRONİK,"Electronic & Electromechanical Systems, Commun...",info@kzmekatronik.com,https://www.kzmekatronik.com,Yeni Sanayi Sit. 27. Sokak No:32 10020 Karesi ...,Trkiye,4.0,4G-04,KZ Mekatronik designs and manufactures innovat...,https://sahaexpo.com/en/katilimci/kz-mekatroni...
1,Longsys,NaN,info@demsay.com,https://www.demsay.com/distributor/longsys,Sahil Mah. Marmara Cad. Çayırlı Sok. No:2/1 Be...,China,2.0,2D-14,"Longsys is engaged in R&D, design, and sales o...",https://sahaexpo.com/en/katilimci/longsys
2,Silicon Power Computer & Communications Inc.,NaN,gracia_castillo@silicon-power.com,https://www.silicon-power.com,"7F, No. 106, Zhouzi St., Neihu District, Taipe...",Chinese Taipei,3.0,3F-32,"Silicon Power, established in February 2003, f...",https://sahaexpo.com/en/katilimci/silicon-powe...
3,BERDAN CİVATA,Machinery and Other Manufacturing Equipment,info@berdancivata.com,https://www.berdancivata.com,MERSİN TARSUS ORG.SAN.BÖL.5. CAD NO:6 HUZURKEN...,Trkiye,3.0,3B-02p,Founded in 1980 in Tarsus by brothers Hasan Şe...,https://sahaexpo.com/en/katilimci/berdan-civata
4,RENİSHAW,"Machinery and Other Manufacturing Equipment, T...",turkiye@renishaw.com,https://www.renishaw.com,Serifali Mah. Turgut Özal Bulvari No:193 Umran...,United Kingdom,1.0,1C-16f,Renishaw is an established world leader in met...,https://sahaexpo.com/en/katilimci/renishaw
...,...,...,...,...,...,...,...,...,...,...
1463,MARS RF,"Communication, Command Control, and Informatio...",info@marsrf.com,https://www.marsrf.com,"No. 1, 3rd Floor, Unit 1, Building 28, Jingye ...",China,3.0,3G-24,Mars RF is the leading manufacturer focused on...,https://sahaexpo.com/en/katilimci/mars-rf
1464,SALT CAM,Material and Material Shaping Technologies,saltcam@saltcam.com,https://www.saltcam.com,Hüseyin Okan Merzeci Mh. Akbelen Bulvarı No:17...,Trkiye,3.0,3B-02k,80 Years of Experience in GlassSalt Cam is one...,https://sahaexpo.com/en/katilimci/salt-cam
1465,LIGHTERPROPELLER,"Composite Material Technologies, Material and ...",info@lighterpropeller.com,https://www.lighterpropeller.com,"Level 11, Menara LGB 1, Jalan Wan Kadir，Taman ...",Malaysia,1.0,1D-12b,"Since 2013, we have been committed to developi...",https://sahaexpo.com/en/katilimci/lighterprope...
1466,DAMISE,"Software and Artificial Intelligence, Informat...",info@damise.com,https://www.damise.com,Çamlık Mah. Şenol Güneş Bulvarı Dinç Sok. No:4...,Trkiye,2.0,2D-01,DAMISE; is Türkiye’s first sector-focused sear...,https://sahaexpo.com/en/katilimci/damise
